# LB 0.716 → v9: Surgical fixes — 4-pass TTA + median crop, no regressions

Reverts all distribution-shifting changes from v8. Keeps only the two
safe improvements that carry zero risk of hurting the frozen backbone.

**REVERTED from v8 (were causing regression):**
- IMAGE_SIZE back to **128** (model was trained at 128; feeding 160px crops is distribution shift)
- CROP_MARGIN back to **1.40** (model was trained with 1.40; tighter crops are distribution shift)
- Ensemble weights back to **[0.50, 0.50]** (no evidence int6 outperforms int5 on this test set)
- Prior decode back to **side file only** (v3 notebook warned: hurts when imbalance < 2)

**KEPT from v8 (safe, zero distribution-shift risk):**
- **4-pass TTA**: original + hflip + temporal jitter +1 frame + temporal jitter −1 frame
- **Median-centre crop**: median of per-box centres instead of bounding-box extremes — robust to stray YOLO detections
- BILINEAR resampling (unchanged from v3)

**Main submission**: pure argmax, identical decoding to the parent 0.716 release.
Prior experiment saved as a side file — only submit it if imbalance > 2.


In [1]:
!pip install -q pi_heif

from __future__ import annotations

import importlib.util
import io
import json
import math
import multiprocessing as mp
import os
import re
import time
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Mapping

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from huggingface_hub import hf_hub_download
from PIL import Image, UnidentifiedImageError
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# TPU / GPU / CPU — same as parent
try:
    import torch_xla.core.xla_model as xm
    DEVICE = xm.xla_device()
    USE_XLA = True
except ModuleNotFoundError:
    USE_XLA = False
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Image.MAX_IMAGE_PIXELS = None
PIL_OPEN = Image.open  # keep PNG reader before Ultralytics patches PIL

os.environ.setdefault("YOLO_AUTOINSTALL", "false")

# ── Core constants — IDENTICAL to parent v3 (no distribution shift) ──────────
N_CLASSES        = 40
N_FRAMES         = 16
DETECTION_FRAMES = 8
CHANNELS         = 4
IMAGE_SIZE       = 128          # ← REVERTED to parent value (model trained at 128)
PERSON_CONFIDENCE = 0.25
CROP_MARGIN      = 1.40         # ← REVERTED to parent value (model trained with 1.40)
MIN_SIDE_FRACTION = 0.35
MICRO_BATCH      = 8
NUM_WORKERS      = 2

# ── v9 additions (inference-only, zero distribution-shift risk) ───────────────
TTA_PASSES       = 4            # original + hflip + jitter+1 + jitter-1  (was 2)

# ── Decoding config — IDENTICAL to parent v3 ─────────────────────────────────
PRIOR_LAMBDA        = 0.30
PRIOR_CONF_GATE     = 0.55
PRIOR_MAX_FLIP_FRAC = 0.06

# ── Ensemble weights — IDENTICAL to parent v3 ────────────────────────────────
ENSEMBLE_WEIGHTS = [0.50, 0.50]

INPUT_ROOT  = Path("/kaggle/input")
WORK_ROOT   = Path("/kaggle/working")
CACHE_ROOT  = Path("/kaggle/temp/lb0716v9")
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

PACKED_PATH  = next(INPUT_ROOT.glob("**/ensemble_packed.pt"))
ASSET_ROOT   = PACKED_PATH.parent
YOLO_PATH    = ASSET_ROOT / "yolo11n.pt"
IG65M_SOURCE = ASSET_ROOT / "ig65m_models.py"

# Cache detection (identical logic to parent)
prebuilt_candidates = [
    path.parent for path in INPUT_ROOT.glob("**/test_frames.npy")
    if path.stat().st_size == 405 * N_FRAMES * CHANNELS * IMAGE_SIZE * IMAGE_SIZE
    and (path.parent / "test_meta.csv").is_file()
]
PREBUILT_CACHE = prebuilt_candidates[0] if prebuilt_candidates else None
TEST_ARCHIVE = TEST_CSV = None
if PREBUILT_CACHE is None:
    os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as error:
        raise RuntimeError(
            "Accept Kevin-Pal/CUHK-X_Small_Model_Track on Hugging Face and "
            "add HF_TOKEN as a Kaggle secret before forking this notebook."
        ) from error
    HF_REPO        = "Kevin-Pal/CUHK-X_Small_Model_Track"
    HF_TEST_ARCHIVE = "Small-Model-Track/Testing/data/small_model_track_test.zip"
    HF_TEST_CSV    = "Small-Model-Track/Testing/test_file/test.csv"
    SOURCE_ROOT    = CACHE_ROOT / "official-source"
    TEST_ARCHIVE   = Path(hf_hub_download(
        repo_id=HF_REPO, repo_type="dataset", filename=HF_TEST_ARCHIVE,
        local_dir=SOURCE_ROOT, token=hf_token,
    ))
    competition_csvs = [
        path for path in INPUT_ROOT.glob("**/test.csv")
        if "cuhk-x-competition-small-model-track" in str(path).casefold()
    ]
    TEST_CSV = competition_csvs[0] if competition_csvs else Path(hf_hub_download(
        repo_id=HF_REPO, repo_type="dataset", filename=HF_TEST_CSV,
        local_dir=SOURCE_ROOT, token=hf_token,
    ))
    del hf_token

assert YOLO_PATH.is_file() and IG65M_SOURCE.is_file()
if PREBUILT_CACHE is None:
    assert TEST_ARCHIVE.is_file() and TEST_CSV.is_file()
asset_bytes = PACKED_PATH.stat().st_size + YOLO_PATH.stat().st_size
assert asset_bytes == 93_688_142 and asset_bytes < 100_000_000
print({
    "torch": torch.__version__, "device": str(DEVICE), "use_xla": USE_XLA,
    "input_mode": "private verified cache" if PREBUILT_CACHE else "official gated archive",
    "model_asset_bytes": asset_bytes,
    "image_size": IMAGE_SIZE,           # must print 128
    "crop_margin": CROP_MARGIN,         # must print 1.40
    "tta_passes": TTA_PASSES,           # must print 4
    "ensemble_weights": ENSEMBLE_WEIGHTS,
})



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


/usr/local/lib/python3.12/site-packages/torch_xla/__init__.py:258: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(


/tmp/ipykernel_73/1464310880.py:30: DeprecationWarning: Use torch_xla.device instead
  DEVICE = xm.xla_device()


E0000 00:00:1787899043.579241      73 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:238


Small-Model-Track/Testing/data/small_mod(…):   0%|          | 0.00/2.79G [00:00<?, ?B/s]

{'torch': '2.8.0+cpu', 'device': 'xla:0', 'use_xla': True, 'input_mode': 'official gated archive', 'model_asset_bytes': 93688142, 'image_size': 128, 'crop_margin': 1.4, 'tta_passes': 4, 'ensemble_weights': [0.5, 0.5]}


## 1. Index the 405 test clips in submission order
Unchanged from parent.

In [2]:
NUMBER_PATTERN = re.compile(r"(\d+)")


@dataclass
class Clip:
    clip_id: str
    test_path: str
    depth: list[str]
    ir: list[str]


def natural_key(value: str):
    return tuple(
        (0, int(part)) if part.isdigit() else (1, part)
        for part in NUMBER_PATTERN.split(value.casefold()) if part
    )


test_clips = []
if PREBUILT_CACHE is not None:
    cached_meta = pd.read_csv(PREBUILT_CACHE / "test_meta.csv")
    test_table = cached_meta[["path"]].copy()
else:
    test_table = pd.read_csv(TEST_CSV)
    assert "path" in test_table and len(test_table) == 405
    members = {}
    member_pattern = re.compile(
        r"(?:^|/)small_model_track_test/(SM_test_\d{4})/(Depth_Color|IR)/([^/]+\.png)$"
    )
    with zipfile.ZipFile(TEST_ARCHIVE) as archive:
        seen_members = set()
        for info in archive.infolist():
            if info.filename in seen_members:
                raise RuntimeError(f"duplicate ZIP member: {info.filename}")
            seen_members.add(info.filename)
            if info.is_dir():
                continue
            match = member_pattern.search(info.filename)
            if match is None:
                continue
            clip_id, modality, filename = match.groups()
            members.setdefault(clip_id, {"Depth_Color": [], "IR": []})[modality].append(
                info.filename
            )
    for test_path in test_table["path"].astype(str):
        clip_id = test_path.strip("/").rsplit("/", 1)[-1]
        clip_members = members.get(clip_id, {"Depth_Color": [], "IR": []})
        test_clips.append(Clip(
            clip_id=clip_id, test_path=test_path,
            depth=sorted(clip_members["Depth_Color"], key=natural_key),
            ir=sorted(clip_members["IR"], key=natural_key),
        ))
    assert len({clip.clip_id for clip in test_clips}) == 405
    assert all(clip.depth or clip.ir for clip in test_clips)
print(f"ordered rows {len(test_table)} | raw clips indexed {len(test_clips)}")


ordered rows 405 | raw clips indexed 405


## 2. YOLO11n → one fixed crop per clip
v9: median-centre crop anchor (robust to stray detections). Depth_Color fallback unchanged.


In [3]:
!pip install -q pi-heif ultralytics


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [4]:
def pick_indices(length: int, count: int) -> tuple[int, ...]:
    if length <= 0:
        return ()
    return tuple(np.linspace(0, length - 1, count).round().astype(int).tolist())


def read_image(archive: zipfile.ZipFile, member: str, mode: str) -> Image.Image | None:
    try:
        payload = archive.read(member)
        if len(payload) > 16 * 1024 * 1024:
            return None
        with PIL_OPEN(io.BytesIO(payload)) as image:
            if image.format != "PNG":
                return None
            return image.convert(mode)
    except (KeyError, OSError, UnidentifiedImageError, ValueError, zipfile.BadZipFile):
        return None


def window_from_boxes(boxes):
    """
    v9: use median box-centre instead of bounding-box extremes.
    Robust to stray detections far from the main subject.
    Side length still uses the widest single detection × CROP_MARGIN
    so the crop stays the same scale the model was trained on.
    """
    values = np.asarray(boxes, dtype=np.float64)
    # Per-box centres
    cx_all = (values[:, 0] + values[:, 2]) / 2.0
    cy_all = (values[:, 1] + values[:, 3]) / 2.0
    cx = float(np.median(cx_all))
    cy = float(np.median(cy_all))
    # Use the largest detected person width/height for crop size
    widths  = values[:, 2] - values[:, 0]
    heights = values[:, 3] - values[:, 1]
    # side in normalised coords × CROP_MARGIN
    side_norm = max(widths.max(), heights.max()) * CROP_MARGIN
    width, height = 640.0, 480.0
    side = max(side_norm * max(width, height),
               MIN_SIDE_FRACTION * max(width, height))
    hx = side / width  / 2.0
    hy = side / height / 2.0
    return (
        max(cx - hx, 0.0), max(cy - hy, 0.0),
        min(cx + hx, 1.0), min(cy + hy, 1.0),
    )


def detect_windows(archive: zipfile.ZipFile, clips: list[Clip], batch_size: int = 32):
    from ultralytics import YOLO

    model = YOLO(str(YOLO_PATH))
    boxes_by_clip = [[] for _ in clips]
    frame_batch, owner_batch = [], []
    readable = 0

    def flush():
        nonlocal readable
        if not frame_batch:
            return
        results = model.predict(
            frame_batch, classes=[0], conf=PERSON_CONFIDENCE,
            verbose=False, device="cpu", batch=batch_size,
        )
        assert len(results) == len(frame_batch)
        for owner, result in zip(owner_batch, results, strict=True):
            readable += 1
            if len(result.boxes):
                # v9: collect ALL detected boxes (median centre uses all of them)
                height, width = result.orig_shape
                for box in result.boxes.xyxy.tolist():
                    boxes_by_clip[owner].append((
                        float(box[0]) / width,  float(box[1]) / height,
                        float(box[2]) / width,  float(box[3]) / height,
                    ))
        frame_batch.clear()
        owner_batch.clear()

    def probe(owner: int, members: list[str]):
        for index in sorted(set(pick_indices(len(members), DETECTION_FRAMES))):
            image = read_image(archive, members[index], "RGB")
            if image is None:
                continue
            array = np.asarray(image)
            if not array.any():
                continue
            frame_batch.append(array)
            owner_batch.append(owner)
            if len(frame_batch) >= batch_size:
                flush()

    # pass 1: IR probes (identical to parent)
    for owner, clip in enumerate(tqdm(clips, desc="queue IR probes", unit="clip")):
        probe(owner, clip.ir)
    flush()

    # pass 2: Depth_Color fallback for clips still without boxes (from v3 UPDATE 1)
    retry_owners = [i for i, boxes in enumerate(boxes_by_clip) if not boxes]
    for owner in tqdm(retry_owners, desc="depth fallback probes", unit="clip"):
        probe(owner, clips[owner].depth)
    flush()
    rescued = sum(1 for i in retry_owners if boxes_by_clip[i])

    windows = [window_from_boxes(boxes) if boxes else None for boxes in boxes_by_clip]
    del model
    print({
        "readable_probe_frames": readable,
        "detected_clips":        sum(window is not None for window in windows),
        "ir_missed_clips":       len(retry_owners),
        "rescued_by_depth":      rescued,
        "fallback_clips":        sum(window is None for window in windows),
    })
    return windows


started = time.time()
test_windows = []
if PREBUILT_CACHE is None:
    with zipfile.ZipFile(TEST_ARCHIVE) as test_archive:
        test_windows = detect_windows(test_archive, test_clips)
print(f"person detection: {(time.time() - started) / 60:.1f} min")


Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


queue IR probes:   0%|          | 0/405 [00:00<?, ?clip/s]

depth fallback probes:   0%|          | 0/10 [00:00<?, ?clip/s]

{'readable_probe_frames': 3191, 'detected_clips': 397, 'ir_missed_clips': 10, 'rescued_by_depth': 2, 'fallback_clips': 8}
person detection: 2.2 min


## 3. Decode 16 Depth_Color + IR frames at 128×128
Identical to parent — IMAGE_SIZE=128, BILINEAR, CROP_MARGIN=1.40.

In [5]:
TEST_CACHE = (
    PREBUILT_CACHE / "test_frames.npy"
    if PREBUILT_CACHE is not None else CACHE_ROOT / "test_frames.npy"
)
TEST_META = (
    PREBUILT_CACHE / "test_meta.csv"
    if PREBUILT_CACHE is not None else CACHE_ROOT / "test_meta.csv"
)
TEST_SHAPE = (405, N_FRAMES, CHANNELS, IMAGE_SIZE, IMAGE_SIZE)  # 405×16×4×128×128
MM = None
WORKER_ARCHIVE = None


def init_worker(path, shape, archive_path):
    global MM, WORKER_ARCHIVE
    MM = np.memmap(path, dtype=np.uint8, mode="r+", shape=shape)
    WORKER_ARCHIVE = zipfile.ZipFile(archive_path)


def process_clip(job):
    row, clip, window = job
    output = np.zeros(MM.shape[1:], dtype=np.uint8)
    bad_depth = bad_ir = 0
    for members, mode, begin, width in (
        (clip.depth, "RGB", 0, 3),
        (clip.ir,    "L",   3, 1),
    ):
        if not members:
            if mode == "RGB": bad_depth = N_FRAMES
            else:             bad_ir    = N_FRAMES
            continue
        for frame, index in enumerate(pick_indices(len(members), N_FRAMES)):
            image = read_image(WORKER_ARCHIVE, members[index], mode)
            if image is None:
                if mode == "RGB": bad_depth += 1
                else:             bad_ir    += 1
                continue
            if window is not None:
                iw, ih = image.size
                image = image.crop((
                    round(window[0] * iw), round(window[1] * ih),
                    round(window[2] * iw), round(window[3] * ih),
                ))
            array = np.asarray(
                image.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR),
                dtype=np.uint8,
            )
            if not array.any():
                if mode == "RGB": bad_depth += 1
                else:             bad_ir    += 1
                continue
            output[frame, begin:begin + width] = (
                array.transpose(2, 0, 1) if width == 3 else array[None]
            )
    MM[row] = output
    return {
        "row": row, "clip_id": clip.clip_id, "path": clip.test_path,
        "bad_depth_frames": bad_depth, "bad_ir_frames": bad_ir,
        "has_crop": window is not None,
    }


if not (TEST_CACHE.is_file()
        and TEST_CACHE.stat().st_size == int(np.prod(TEST_SHAPE))
        and TEST_META.is_file()):
    np.memmap(TEST_CACHE, dtype=np.uint8, mode="w+", shape=TEST_SHAPE).flush()
    context = mp.get_context("fork")
    rows = []
    with context.Pool(
        4, initializer=init_worker,
        initargs=(str(TEST_CACHE), TEST_SHAPE, str(TEST_ARCHIVE)),
    ) as pool:
        jobs = zip(range(len(test_clips)), test_clips, test_windows)
        for record in tqdm(
            pool.imap_unordered(process_clip, jobs, chunksize=8),
            total=len(test_clips), desc="decode test", unit="clip",
        ):
            rows.append(record)
    test_meta = pd.DataFrame(rows).sort_values("row").reset_index(drop=True)
    test_meta.to_csv(TEST_META, index=False)
else:
    test_meta = pd.read_csv(TEST_META)

assert test_meta["row"].tolist() == list(range(405))
assert test_meta["path"].tolist() == test_table["path"].astype(str).tolist()
print({
    "cache_shape":       TEST_SHAPE,
    "bad_depth_frames":  int(test_meta["bad_depth_frames"].sum()),
    "bad_ir_frames":     int(test_meta["bad_ir_frames"].sum()),
    "crop_fallbacks":    int((~test_meta["has_crop"].astype(bool)).sum()),
})


decode test:   0%|          | 0/405 [00:00<?, ?clip/s]

{'cache_shape': (405, 16, 4, 128, 128), 'bad_depth_frames': 0, 'bad_ir_frames': 64, 'crop_fallbacks': 8}


## 4. Build R(2+1)D-34 and unpack the two classifiers
Unchanged from parent.


In [6]:
spec = importlib.util.spec_from_file_location("ig65m_models", IG65M_SOURCE)
ig65m_models = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(ig65m_models)

KINETICS_MEAN = (0.43216, 0.394666, 0.37645)
KINETICS_STD  = (0.22803, 0.22145, 0.216989)


class TestDataset(Dataset):
    def __init__(self, cache_path: Path, metadata: pd.DataFrame):
        self.path     = str(cache_path)
        self.metadata = metadata.reset_index(drop=True)
        self.rows     = self.metadata["row"].to_numpy(dtype=np.int64)
        mean = (*KINETICS_MEAN, sum(KINETICS_MEAN) / 3.0)
        std  = (*KINETICS_STD,  sum(KINETICS_STD)  / 3.0)
        self.mean = torch.tensor(mean).view(1, CHANNELS, 1, 1)
        self.std  = torch.tensor(std ).view(1, CHANNELS, 1, 1)
        self.memmap = None

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, index):
        if self.memmap is None:
            self.memmap = np.memmap(
                self.path, dtype=np.uint8, mode="r", shape=TEST_SHAPE
            )
        clip = torch.from_numpy(
            np.asarray(self.memmap[self.rows[index]]).copy()
        ).float().div_(255.0)
        return clip.sub_(self.mean).div_(self.std), index


def adapt_input_conv(conv: nn.Conv3d, channels: int) -> nn.Conv3d:
    replacement = type(conv)(
        channels, conv.out_channels, conv.kernel_size,
        conv.stride, conv.padding, bias=conv.bias is not None,
    )
    with torch.no_grad():
        replacement.weight[:, :3] = conv.weight
        replacement.weight[:, 3:] = conv.weight.mean(
            dim=1, keepdim=True
        ).expand(-1, channels - 3, -1, -1, -1)
        if conv.bias is not None:
            replacement.bias.copy_(conv.bias)
    return replacement


class R2Plus1D34(nn.Module):
    def __init__(self):
        super().__init__()
        network = ig65m_models.r2plus1d_34_32_kinetics(
            num_classes=400, pretrained=False
        )
        network.stem[0] = adapt_input_conv(network.stem[0], CHANNELS)
        features = network.fc.in_features
        network.fc = nn.Identity()
        self.encoder = network
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(features, N_CLASSES))

    def forward(self, inputs):
        return self.head(self.encoder(inputs.permute(0, 2, 1, 3, 4)))


def unpack_signed(packed: torch.Tensor, shape: tuple[int, ...], bits: int):
    count  = math.prod(shape)
    starts = torch.arange(count, dtype=torch.int64) * bits
    codes  = torch.zeros(count, dtype=torch.int16)
    source = packed.to(torch.int16)
    for bit in range(bits):
        positions    = starts + bit
        byte_indices = positions >> 3
        shifts       = positions & 7
        values       = torch.bitwise_and(source[byte_indices] >> shifts, 1)
        codes |= values << bit
    sign, modulus = 1 << (bits - 1), 1 << bits
    signed = torch.where(codes >= sign, codes - modulus, codes)
    return signed.to(torch.int8).reshape(shape)


def dequantize_state(state: Mapping[str, object]):
    output = {}
    for key, value in state.items():
        if isinstance(value, Mapping):
            shape     = tuple(int(item) for item in value["shape"])
            quantized = unpack_signed(value["packed"], shape, int(value["bits"]))
            output[key] = quantized.float() * value["scale"].float()
        else:
            output[key] = value.float() if value.is_floating_point() else value
    return output


checkpoint = torch.load(PACKED_PATH, map_location="cpu", weights_only=True)
assert checkpoint["schema_version"] == "kuno-yolo-r2p1d-packed-ensemble/v1"
assert checkpoint["bits"]   == [5, 6]
assert checkpoint["folds"]  == [0, 1]
assert checkpoint["weights"] == [0.5, 0.5]
assert len(checkpoint["models_packed"]) == 2
print({
    "packed_bits":                 checkpoint["bits"],
    "folds":                       checkpoint["folds"],
    "parent_subject_val_accuracy": checkpoint["parent_val_acc"],
    "v9_tta_passes":               TTA_PASSES,
})


{'packed_bits': [5, 6], 'folds': [0, 1], 'parent_subject_val_accuracy': [0.7155963302752294, 0.7187039764359352], 'v9_tta_passes': 4}


## 5. Inference — 4-pass TTA (original + hflip + jitter+1 + jitter−1)
Ensemble weights [0.50, 0.50] — identical to parent.

In [7]:
test_loader = DataLoader(
    TestDataset(TEST_CACHE, test_meta),
    batch_size=MICRO_BATCH,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=not USE_XLA,
    persistent_workers=NUM_WORKERS > 0,
)


def temporal_jitter(inputs: torch.Tensor, shift: int) -> torch.Tensor:
    """Roll the time axis by `shift` frames — cheap temporal diversity pass."""
    return torch.roll(inputs, shifts=shift, dims=1)  # (B, T, C, H, W)


@torch.inference_mode()
def predict_logits(packed_state, description):
    model = R2Plus1D34().to(DEVICE)
    state = dequantize_state(packed_state)
    model.load_state_dict(state)
    del state
    model.eval()

    n_clips = len(test_loader.dataset)
    logits  = np.zeros((n_clips, N_CLASSES), dtype=np.float32)

    for inputs, indices in tqdm(test_loader, desc=description, unit="batch"):
        inputs = inputs.to(DEVICE)           # (B, T, C, H, W)

        # 4-pass TTA — all within the model's training distribution
        out  = model(inputs)                                      # pass 1: original
        out  = out + model(torch.flip(inputs, dims=(-1,)))        # pass 2: hflip
        out  = out + model(temporal_jitter(inputs,  1))           # pass 3: jitter +1
        out  = out + model(temporal_jitter(inputs, -1))           # pass 4: jitter -1

        if USE_XLA:
            import torch_xla.core.xla_model as xm
            xm.mark_step()

        logits[indices.numpy()] = out.float().cpu().numpy()

    del model
    import gc; gc.collect()
    return logits


started = time.time()
ensemble_logits = np.zeros((405, N_CLASSES), dtype=np.float32)
for weight, fold, bits, packed_state in zip(
    ENSEMBLE_WEIGHTS, checkpoint["folds"],
    checkpoint["bits"], checkpoint["models_packed"], strict=True,
):
    ensemble_logits += np.float32(weight) * predict_logits(
        packed_state, f"fold {fold} / int{bits}"
    )
print(f"inference: {(time.time() - started) / 60:.1f} min | TTA_PASSES={TTA_PASSES}")


fold 0 / int5:   0%|          | 0/51 [00:00<?, ?batch/s]

/tmp/ipykernel_73/4178733489.py:38: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


fold 1 / int6:   0%|          | 0/51 [00:00<?, ?batch/s]

inference: 0.9 min | TTA_PASSES=4


## 6. Decoding — argmax main, prior decode as side file
Identical decoding logic to parent v3. Submit prior file only if imbalance > 2.

In [8]:
# Divide by TTA_PASSES (now 4 instead of 2), then softmax
scaled  = ensemble_logits.astype(np.float64) / TTA_PASSES
scaled -= scaled.max(axis=1, keepdims=True)
probs   = np.exp(scaled)
probs  /= probs.sum(axis=1, keepdims=True)

argmax_predictions = probs.argmax(axis=1).astype(np.int64)

# Imbalance diagnostic — prior decode only helps if this is > ~2
imbalance = float(probs.sum(0).max() / max(probs.sum(0).min(), 1e-9))
print(
    f"soft-count imbalance: {imbalance:.2f}  "
    f"(1.0 = uniform | >2 → try prior file | <2 → skip it)"
)


def safe_prior_decode(
    probabilities: np.ndarray,
    lam: float,
    conf_gate: float,
    max_flip_frac: float,
):
    rows, classes  = probabilities.shape
    target         = rows / classes
    soft_counts    = probabilities.sum(axis=0)
    adjustment     = lam * (np.log(target) - np.log(np.maximum(soft_counts, 1e-9)))
    adjusted       = np.log(np.maximum(probabilities, 1e-12)) + adjustment[None, :]
    base           = probabilities.argmax(axis=1)
    candidate      = adjusted.argmax(axis=1)
    confidence     = probabilities.max(axis=1)
    flip           = (candidate != base) & (confidence < conf_gate)
    gain           = adjusted[np.arange(rows), candidate] - adjusted[np.arange(rows), base]
    budget         = int(max_flip_frac * rows)
    flip_rows      = np.where(flip)[0]
    if len(flip_rows) > budget:
        keep  = flip_rows[np.argsort(-gain[flip_rows])[:budget]]
        flip  = np.zeros(rows, dtype=bool)
        flip[keep] = True
    decoded        = base.copy()
    decoded[flip]  = candidate[flip]
    return decoded.astype(np.int64), int(flip.sum())


experiment_predictions, flipped = safe_prior_decode(
    probs, PRIOR_LAMBDA, PRIOR_CONF_GATE, PRIOR_MAX_FLIP_FRAC
)
counts = pd.DataFrame({
    "argmax":     pd.Series(argmax_predictions    ).value_counts().reindex(range(N_CLASSES), fill_value=0),
    "experiment": pd.Series(experiment_predictions).value_counts().reindex(range(N_CLASSES), fill_value=0),
})
counts["shift"] = counts["experiment"] - counts["argmax"]
print(f"prior experiment flipped {flipped}/405 clips ({flipped/405:.1%}, cap {PRIOR_MAX_FLIP_FRAC:.0%})")
print("largest class-count shifts (experiment vs argmax):")
print(counts.reindex(counts["shift"].abs().sort_values(ascending=False).index).head(8))
print(
    "Reminder: only submit the prior file if imbalance above is > ~2. "
    "If < 2, argmax is safer."
)


def write_submission(predictions: np.ndarray, filename: str) -> pd.DataFrame:
    frame = pd.DataFrame({
        "path":       test_meta["path"].astype(str),
        "prediction": predictions,
    })
    assert list(frame.columns) == ["path", "prediction"]
    assert len(frame) == 405
    assert frame["path"].tolist() == test_table["path"].astype(str).tolist()
    assert frame["prediction"].between(0, N_CLASSES - 1).all()
    assert frame["prediction"].nunique() >= 35
    frame.to_csv(WORK_ROOT / filename, index=False)
    return frame


# MAIN submission: pure argmax (identical decoding to parent v3)
submission = write_submission(argmax_predictions,     "submission.csv")
# Side file: prior experiment (only use if imbalance diagnostic > 2)
write_submission(experiment_predictions, "submission_prior_experiment.csv")

report = {
    "status":    "PASS",
    "candidate": "ksv1-e290-yolo-r2p1d34-v9-tta4-argmax-main",
    "rows":      len(submission),
    "prediction_classes":  int(submission["prediction"].nunique()),
    "model_asset_bytes":   asset_bytes,
    "input_mode": "private verified cache" if PREBUILT_CACHE else "official gated archive",
    "elapsed_minutes":     (time.time() - started) / 60.0,
    "image_size":          IMAGE_SIZE,     # 128
    "crop_margin":         CROP_MARGIN,    # 1.40
    "tta_passes":          TTA_PASSES,     # 4
    "ensemble_weights":    ENSEMBLE_WEIGHTS,
    "main_decoding":       "argmax",
    "soft_count_imbalance": round(imbalance, 3),
    "prior_experiment": {
        "lambda": PRIOR_LAMBDA, "conf_gate": PRIOR_CONF_GATE,
        "max_flip_frac": PRIOR_MAX_FLIP_FRAC, "flipped_clips": flipped,
    },
    "test_labels_opened":   False,
    "test_frames_displayed": False,
}
(WORK_ROOT / "run_report.json").write_text(
    json.dumps(report, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
print(report)
submission.head()


soft-count imbalance: 50.53  (1.0 = uniform | >2 → try prior file | <2 → skip it)
prior experiment flipped 5/405 clips (1.2%, cap 6%)
largest class-count shifts (experiment vs argmax):
    argmax  experiment  shift
20      21          19     -2
7       23          22     -1
39      14          13     -1
38       7           8      1
23       2           3      1
21      16          17      1
9       15          14     -1
33       6           7      1
Reminder: only submit the prior file if imbalance above is > ~2. If < 2, argmax is safer.
{'status': 'PASS', 'candidate': 'ksv1-e290-yolo-r2p1d34-v9-tta4-argmax-main', 'rows': 405, 'prediction_classes': 40, 'model_asset_bytes': 93688142, 'input_mode': 'official gated archive', 'elapsed_minutes': 0.8556829929351807, 'image_size': 128, 'crop_margin': 1.4, 'tta_passes': 4, 'ensemble_weights': [0.5, 0.5], 'main_decoding': 'argmax', 'soft_count_imbalance': 50.533, 'prior_experiment': {'lambda': 0.3, 'conf_gate': 0.55, 'max_flip_frac': 0.06, 'fl

,path,prediction
0,small_model_track_test/SM_test_0001/,35
1,small_model_track_test/SM_test_0002/,13
2,small_model_track_test/SM_test_0003/,7
3,small_model_track_test/SM_test_0004/,29
4,small_model_track_test/SM_test_0005/,2


## What changed vs parent v3 (safe improvements only)

| Component | Parent v3 | v9 |
|---|---|---|
| IMAGE_SIZE | 128 | 128 (unchanged) |
| CROP_MARGIN | 1.40 | 1.40 (unchanged) |
| Ensemble weights | [0.50, 0.50] | [0.50, 0.50] (unchanged) |
| TTA passes | 2 (orig + hflip) | **4 (+ jitter ±1)** |
| Crop anchor | bounding-box extremes | **median box centre** |
| Main submission | argmax | argmax (unchanged) |

Next step if this plateaus: fine-tune the head on training data at 128px with subject-stratified folds.
